<a href="https://colab.research.google.com/github/pgordin/OptDisc2026/blob/main/Grafy3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algorytmy optymalizacji dyskretnej; laboratoria

Notatnik zawiera definicje klasy *Graph*, zawierającą funkcje przedstawiana na laboaratorium.

Importowanie pakietów

In [1]:
import numpy as np
from random import random, seed

## Proste funkcje grafowe z pierwszych zajęć

In [2]:
def print_matrix(vertices, matrix):
  """
  Wypisuje na ekranie graf podany jako macierz sąsiedztwa
  """
  n = len(matrix)
  if (vertices is None) or (len(vertices) != n):
    vv = range(1, n+1)
  else:
    vv = vertices
  for i in range(n):
    print(vv[i], ':', end='')
    for j in range(n):
      if (matrix[i][j]):
        print(" ", vv[j], end="")
    print("")

In [3]:
def print_dict(graph):
  """
  Wypisuje na ekranie graf podany jako słownik (list) sąsiedztwa
  """
  for v in graph:
    print(v, ':', end="")
    for u in graph[v]:
      print(" ", u, end="")
    print("")


## Klasa *Graph*

In [4]:
class Graph:
    def __init__(self, graph=None):
        if graph is None:
            graph = {}
        self.graph = graph

    # inicjalizator ze słownika
    @classmethod
    def from_dict(cls, graph):
        return cls(graph)

    # inicjalizator z macierzy
    @classmethod
    def from_matrix(cls, matrix, vertices = None):
        if (vertices is None) or (len(vertices) != len(matrix)):
            vertices = [*range(1, len(matrix) + 1)]
        return cls.from_dict(cls._matrix_to_dict(matrix, vertices))

    # dwie prywatne metody macierz <-> słownik
    def _matrix_to_dict(matrix, vertices: list) -> dict:
        """
        Zamienia graf podany jako macierz sąsiedztwa na słownik sąsiedztwa.
        """
        res_dict = {}
        for i, v in enumerate(vertices):
            neighbours = [vertices[j] for j, edge in enumerate(matrix[i]) if edge]
            res_dict[v] = neighbours
        return res_dict

    def _dict_to_matrix(self, _dict: dict) -> np.array:
        """
        Zamienia graf podany jako słownik sąsiedztwa na macierz sąsiedztwa.
        """
        n = len(_dict)
        vertices = [*_dict.keys()]
        matrix = np.zeros(shape = (n, n), dtype=int)
        for u,v in [
            (vertices.index(u), vertices.index(v))
            for u, row in _dict.items() for v in row
        ]:
            matrix[u][v] += 1
        return matrix

    def vertices(self) -> list:
        """
        Zwraca listę wierzchołków grafu.
        """
        return [*self.graph.keys()]

    def matrix(self) -> np.array:
        """
        Zwraca macierz sąsiedztwa grafu.
        """
        return self._dict_to_matrix(self.graph)

    # przedefiniowania sposobu wyświetlania grafów
    def __str__(self):
        res = ""
        for v in self.graph:
            res += f"{v}:"
            for u in self.graph[v]:
                res += f" {u}"
            res += "\n"
        return res

    # Poniższe dostajemy za darmo z powyższego
    def to_neighbourlist(self, filename: str):
        """
        Zapisuje graf podany jako słownik (list) sąsiedztwa do pliku (w formie listy sąsiedztwa).
        Zmienna filename zawiera pełną ścieżkę pliku
        """
        file = open(filename, "w")  # otwarcie pliku tekstowego do zapisu
        file.write(str(self))
        file.close()

    # Modyfikacje grafów
    def add_vertex(self, vertex):
        """
        Dodaje wierzchołek do grafu
        """
        if vertex not in self.graph:
            self.graph[vertex] = []

    def del_vertex(self, vertex):
        """
        Usuwa wierzchołek z grafu
        """
        if vertex in self.graph:
            self.graph.pop(vertex)
            for u in self.graph:
                if vertex in self.graph[u]:
                    self.graph[u].remove(vertex)

    def add_arc(self, arc):
        """
        Dodaje łuk (skierowany, podany jako para wierzchołków) do grafu
        """
        u, v = arc
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)

    def add_edge(self, edge: list):
        """
        Dodaje krawędź (podaną jako para wierzchołków) do grafu
        Rozpatrujemy grafy proste, nieskierowane
        """
        u, v = edge
        if u == v:
            raise ValueError("Pętle nie są dopuszczalne!")
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)
        if u not in self.graph[v]:
            self.graph[v].append(u)

    # czytanie z plików
    @staticmethod
    def from_edges(filename: str, directed = 0):
        """
        Tworzy graf na podstawie pliku z łukami/krawędziami.
        Opis łuku/krawędzi to dwa słowa lub wierzchołka (jedno słowo).
        Nadmiarowe słowa są ignorowane.
        Zmienna filename zawiera pełną ścieżkę pliku
        """
        graph = Graph()
        file = open(filename, "r")          # otwarcie pliku do odczytu
        for line in file:                   # dla każdej linii w pliku
          words = line.strip().split()      # rozdziel linię na słowa
          if len(words) == 1:               # jedno słowo - opis wierzchołka
            graph.add_vertex(words[0])
          elif len(words) >= 2:             # conajmnej 2 słowa - opis krawędzi/łuku
            if directed:
              graph.add_arc([words[0], words[1]])
            else:
              graph.add_edge([words[0], words[1]])
        file.close()
        return graph

    @staticmethod
    def random_graph(n: int, p: float):
        """
        Tworzy losowy graf nieskierowany G(n,p)
        """
        rand_graph = Graph()
        for i in range(1, n + 1):
            rand_graph.add_vertex(i)
            for j in range(1, i):
                if random() < p:
                    rand_graph.add_edge([i, j])
        return rand_graph

    @staticmethod
    def cycle(n: int):
        """
        Tworzy graf cykliczny o n wierzchołkach
        """
        cycle = Graph()
        for i in range(n-1):
          cycle.add_edge([i+1, i+2])
        cycle.add_edge([1, n])
        return cycle

# Testowanie funkcji

## Wczytywanie i zapis grafów

In [5]:
%%writefile edges.txt
a b
a c
b d
c e
f

Overwriting edges.txt


In [6]:
%cat edges.txt

a b
a c
b d
c e
f


In [7]:
graph1 = Graph.from_edges('edges.txt')

In [8]:
print(graph1)

a: b c
b: a d
c: a e
d: b
e: c
f:



In [9]:
digraph1 = Graph.from_edges('edges.txt', directed=1)
print(digraph1)

a: b c
b: d
c: e
d:
e:
f:



In [10]:
!wget https://raw.githubusercontent.com/pgordin/OptDisc2026/refs/heads/main/weighted0.txt

--2026-04-19 22:11:07--  https://raw.githubusercontent.com/pgordin/OptDisc2026/refs/heads/main/weighted0.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 114 [text/plain]
Saving to: ‘weighted0.txt.1’

weighted0.txt.1     100%[===================>]     114  --.-KB/s    in 0s      

2026-04-19 22:11:07 (1.06 MB/s) - ‘weighted0.txt.1’ saved [114/114]



In [11]:
%cat weighted0.txt

A B 3
A E 10
B C 26
B D 12
C D 17
C F 13
C G 14
D E 7
D F 15
E F 8
E H 4
F G 9
F H 6
G H 16
G I 11


In [12]:
graph2 = Graph.from_edges('weighted0.txt')
print(graph2)

A: B E
B: A C D
E: A D F H
C: B D F G
D: B C E F
F: C D E G H
G: C F H I
H: E F G
I: G



In [13]:
Graph.to_neighbourlist(graph2, 'graph2.txt')

In [14]:
%cat graph2.txt

A: B E
B: A C D
E: A D F H
C: B D F G
D: B C E F
F: C D E G H
G: C F H I
H: E F G
I: G


In [15]:
seed(2026)  # dla powtarzalności
graph3 = Graph.random_graph(10, 1/3)

In [16]:
print(graph3)

1: 2
2: 1 4 7 8 9 10
3: 4 8
4: 2 3
5: 6 9
6: 5
7: 2
8: 2 3
9: 2 5 10
10: 2 9



In [17]:
print(Graph.cycle(6))

1: 2 6
2: 1 3
3: 2 4
4: 3 5
5: 4 6
6: 5 1



## Pierwsze zajęcia

In [18]:
vertices = ['a', 'b', 'c', 'd']
matrix = np.array([[0,1,1,1],[1,0,0,0],[1,0,0,0],[1,0,0,0]])

In [19]:
print(vertices)

['a', 'b', 'c', 'd']


In [20]:
print(matrix)

[[0 1 1 1]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]


In [21]:
print_matrix(vertices, matrix)

a :  b  c  d
b :  a
c :  a
d :  a


In [22]:
print_matrix(None, matrix)

1 :  2  3  4
2 :  1
3 :  1
4 :  1


In [23]:
print_matrix(['a','b'], matrix)

1 :  2  3  4
2 :  1
3 :  1
4 :  1


In [24]:
graph = {
    'a': ['b', 'c', 'd'],
    'b': ['a'],
    'c': ['a'],
    'd': ['a']
}

In [25]:
print(graph)

{'a': ['b', 'c', 'd'], 'b': ['a'], 'c': ['a'], 'd': ['a']}


In [26]:
print_dict(graph)

a :  b  c  d
b :  a
c :  a
d :  a


In [27]:
graph0 = Graph(graph)
print(graph0)

a: b c d
b: a
c: a
d: a



In [28]:
graph0.add_vertex('e')

In [29]:
graph0.add_edge(('a', 'f'))

In [30]:
print(graph0)

a: b c d f
b: a
c: a
d: a
e:
f: a



In [31]:
graph0.add_edge(('e', 'f'))

In [32]:
graph0.add_edge(('f', 'f'))

ValueError: Pętle nie są dopuszczalne!

In [33]:
graph0.add_arc(('f', 'f'))

In [34]:
print(graph0)

a: b c d f
b: a
c: a
d: a
e: f
f: a e f

